# Time2Success Dataset Collection (RynnValue-format)
Collects video + instruction + timestamp-derived labels from checkpoints of the trained peg-insert-side policy. Data collection only — no model training in this notebook.

## 1. Setup and sanity check

In [1]:
import gymnasium as gym
import metaworld

env = gym.make("Meta-World/MT1", env_name="peg-insert-side-v3", render_mode="rgb_array")
env.reset(seed=0)

# Retrieve simulation timestep
dt = getattr(env.unwrapped, "dt", env.unwrapped.model.opt.timestep)
print("env.dt:", dt)

env.close()

env.dt: 0.0125


d:\Miniconda3\envs\duke_rob\lib\site-packages\gymnasium\utils\passive_env_checker.py:34: UserWarning: WARN: A Box observation space maximum and minimum values are equal.
  logger.warn("A Box observation space maximum and minimum values are equal.")
d:\Miniconda3\envs\duke_rob\lib\site-packages\gymnasium\utils\passive_env_checker.py:157: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


## 2. Env builder + constants

In [2]:
import gymnasium as gym
import metaworld

TASK_NAME = "peg-insert-side-v3"
SUCCESS_KEY = "success"  # verified earlier in the training notebook's flag-check
INSTRUCTION = "Insert the peg into the box from the side."
EMBODIMENT = "Sawyer, MuJoCo simulation, peg-insert-side-v3"

def make_env(seed=0, render_mode=None):
    return gym.make("Meta-World/MT1", env_name=TASK_NAME, seed=seed, render_mode=render_mode)

# Confirm dt on a fresh instance, store as a constant used throughout
_probe_env = make_env(seed=0, render_mode="rgb_array")
DT = getattr(_probe_env.unwrapped, "dt", _probe_env.unwrapped.model.opt.timestep)
_probe_env.close()
print("DT (seconds per step):", DT)

DT (seconds per step): 0.0125


## 3. Output paths

In [3]:
import os

OUTPUT_DIR = "./data/rynn_format_dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Writing dataset to:", os.path.abspath(OUTPUT_DIR))

Writing dataset to: e:\@ML_Projects\Peg_insertion\data\rynn_format_dataset


## 4. Collection + save functions

In [6]:
import numpy as np
import json
import imageio

# stop collection shortly after success, to avoid dead tail
def collect_segment(model, env, deterministic=True, max_steps=500,
                     extra_frames_after_success=30):
    obs, _ = env.reset()
    frames = []
    success_step = None
    frames_since_success = 0
    for t in range(max_steps):
        frames.append(env.render())
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, reward, terminated, truncated, info = env.step(action)
        if info.get(SUCCESS_KEY, 0) and success_step is None:
            success_step = t
        if success_step is not None:
            frames_since_success += 1
            if frames_since_success >= extra_frames_after_success:
                break  # stop shortly after success — no dead tail
        if terminated or truncated:
            break
    return frames, success_step

def save_segment(segment_id, frames, success_step, source_tag):
    seg_dir = os.path.join(OUTPUT_DIR, segment_id)
    os.makedirs(seg_dir, exist_ok=True)

    video_path = os.path.join(seg_dir, "video.mp4")
    imageio.mimsave(video_path, frames, fps=int(round(1 / DT)))

    if success_step is not None:
        t_G = success_step * DT
    else:
        t_G = None

    frame_records = []
    for i, _ in enumerate(frames):
        t_i = i * DT
        if t_G is not None:
            v_star = max(0.0, t_G - t_i)
        else:
            v_star = None
        frame_records.append(dict(frame_idx=i, timestamp_sec=round(t_i, 4),
                                    absolute_temporal_distance_sec=v_star))

    metadata = dict(
        segment_id=segment_id,
        instruction=INSTRUCTION,
        embodiment=EMBODIMENT,
        source_policy=source_tag,
        success=success_step is not None,
        completion_cutoff_sec=t_G,
        num_frames=len(frames),
        dt_sec=DT,
        frames=frame_records,
    )
    with open(os.path.join(seg_dir, "metadata.json"), "w") as f:
        json.dump(metadata, f, indent=2)

    return metadata

## 5. Smoke test — run this before the full collection
Same reasoning as the training smoke test: confirms the loop, video writing, and JSON writing all work before committing to a longer run.

In [7]:
from stable_baselines3 import SAC

_smoke_model = SAC.load("./checkpoints/peg_insert_side/sac_peg_insert_final")
_smoke_env = make_env(seed=0, render_mode="rgb_array")

frames, success_step = collect_segment(_smoke_model, _smoke_env, deterministic=True)
meta = save_segment("seg_smoke_test", frames, success_step, "smoke_test")
_smoke_env.close()

print("Smoke test segment saved. success:", meta["success"], "num_frames:", meta["num_frames"])

Smoke test segment saved. success: True num_frames: 79


## 6. Full collection across checkpoints (no noise — checkpoints alone give success/failure diversity)

In [8]:
import glob

checkpoint_paths = sorted(glob.glob("./checkpoints/peg_insert_side/sac_peg_insert_*.zip"))
final_ckpt = "./checkpoints/peg_insert_side/sac_peg_insert_final"

print(f"Found {len(checkpoint_paths)} numbered checkpoints")

# Spread across training progress rather than every single checkpoint
selected_checkpoints = []
if checkpoint_paths:
    selected_checkpoints = [
        checkpoint_paths[0],
        checkpoint_paths[len(checkpoint_paths)//2],
        checkpoint_paths[-1],
    ]
selected_checkpoints.append(final_ckpt)
print("Using checkpoints:", selected_checkpoints)

Found 8 numbered checkpoints
Using checkpoints: ['./checkpoints/peg_insert_side\\sac_peg_insert_100000.zip', './checkpoints/peg_insert_side\\sac_peg_insert_500000.zip', './checkpoints/peg_insert_side\\sac_peg_insert_final.zip', './checkpoints/peg_insert_side/sac_peg_insert_final']


In [9]:
EPISODES_PER_CHECKPOINT = 5  # smoke-scale first; raise once this runs cleanly end-to-end

all_metadata = []
seg_counter = 0

for ckpt in selected_checkpoints:
    model = SAC.load(ckpt)
    source_tag = os.path.basename(ckpt)
    for ep_idx in range(EPISODES_PER_CHECKPOINT):
        env = make_env(seed=ep_idx, render_mode="rgb_array")
        frames, success_step = collect_segment(model, env, deterministic=True)
        env.close()

        segment_id = f"seg_{seg_counter:05d}"
        meta = save_segment(segment_id, frames, success_step, source_tag)
        all_metadata.append(meta)
        seg_counter += 1

    n_success = sum(m["success"] for m in all_metadata if m["source_policy"] == source_tag)
    print(f"{source_tag}: {n_success}/{EPISODES_PER_CHECKPOINT} succeeded, {seg_counter} segments total so far")

sac_peg_insert_100000.zip: 0/5 succeeded, 5 segments total so far
sac_peg_insert_500000.zip: 5/5 succeeded, 10 segments total so far
sac_peg_insert_final.zip: 5/5 succeeded, 15 segments total so far
sac_peg_insert_final: 5/5 succeeded, 20 segments total so far


## 7. Dataset index

In [ ]:
with open(os.path.join(OUTPUT_DIR, "dataset_index.json"), "w") as f:
    json.dump(dict(
        total_segments=len(all_metadata),
        num_successful=sum(m["success"] for m in all_metadata),
        num_failed=sum(not m["success"] for m in all_metadata),
        sources=sorted(set(m["source_policy"] for m in all_metadata)),
    ), f, indent=2)

print(f"Dataset complete: {len(all_metadata)} segments in {OUTPUT_DIR}")

## 8. Next: once smoke test + full run look correct
Bump `EPISODES_PER_CHECKPOINT` up (e.g. 20–30) and re-run section 6 to build out the full dataset. Check disk usage as you go — video files add up.